# V10-clean: 누수 없는 암종 세부 분류

- 작성: 민준 · 2026-11-10 · 설계서: `docs/specs/2026-11-10-v10-clean-design.md` · 쉬운 설명: `v10_설명.md`
- 목표: **test를 학습에 쓰지 않고** 26개 암종(Macro F1)을 분류한다.
- 모델: LightGBM + 로지스틱 회귀(0.5 : 0.5) / 추가 정보: Geneformer 유전자 임베딩(LR 전용), 변이 시그니처, 쌍둥이 규칙

## 지금까지 결과 (11/10)

| 실험 | 바꾼 것 | CV Macro F1 | Public LB |
|---|---|---|---|
| `baseline` | 기본 피처 | 0.4243 | – |
| `gf_lr` | + Geneformer V1 임베딩 (LR에만) | 0.4343 | – |
| `gf2_lr` | Geneformer V2-104M으로 교체 | 0.4441 | – |
| `gf2_sig` (현재 기본값) | + 변이 시그니처 + 쌍둥이 규칙 | 0.4956 (규칙 포함 시뮬레이션 ≈0.538) | **0.4172** |

⚠️ **CV와 LB 차이가 크다 (0.50 vs 0.42).** 원인: test의 약 35%(879행)가 train과 **다른 표기 방식**이다.
- 종결 코돈을 `*` 대신 `X`로 쓴다 (train에는 `X`가 한 번도 없음)
- 한 칸에 변이가 여러 개 적힌 경우가 많다 (칸당 1.86개, train은 1.19개) → 변이 수가 2배쯤 많아 보인다
- `del`·`ins` 표기가 있다 (train에는 없음)
- 그 결과 이 행들의 61%를 STES(변이가 많은 암)로 예측했다. 다음 할 일: 표기를 train과 같게 읽는 파서 규칙.

## 왜 처음부터 다시 만들었나 (V6~V9에서 반복된 실수)

| # | 실수 | 이번에 막는 방법 |
|---|---|---|
| 1 | test 예측 분포·test 변이 통계를 보고 모델을 고쳤다 (→ 데이터 누수 판정) | test는 **변환만** 한다 |
| 2 | 제출 1번으로 큰 결론을 냈다 | 판단은 로컬 CV로만, 제출은 확인용 |
| 3 | 검증 데이터로 early stopping, fold 밖에서 단어장 생성 → 로컬 점수가 부풀려짐 | early stopping·단어장 모두 fold의 학습 부분 안에서만 |
| 4 | 5개 모델 슈퍼앙상블, 행 정규화, 승수 보정… 복잡해질수록 나빠짐 | 모델 2개, 후처리는 argmax + 쌍둥이 규칙(8-1절) 하나 |
| 5 | 실험 스크립트가 여기저기 흩어져 재현 불가 | 이 노트북 하나 + `out/<실험명>/` |
| 6 | 팀원마다 검증 분할이 달라 점수 비교 불가 | 팀 공통 `folds.csv` |

## 데이터 누수 방지 원칙 (대회 Data Leakage 안내 기준)

1. **test는 변환만 한다.** 어떤 유전자·변이를 피처로 쓸지, 스케일러, 가중치는 전부 train에서 정한다.
2. **test의 각 행은 혼자 처리된다.** 한 환자를 처리할 때 다른 환자 정보를 쓰지 않는다 (7번 섹션에서 자동 검사).
3. **CV에서도 똑같이 한다.** 각 fold의 단어장은 그 fold의 학습 부분으로만 만든다.
4. **test를 보고 정한 숫자가 없다.** 클래스 비율 맞추기, 승수 보정, test 기반 유전자 목록을 쓰지 않는다.

`HGVS_LEGACY`(`X`를 종결 코돈으로 읽기)는 아직 **꺼져 있다**. test의 35%가 `X` 표기를 쓰므로 켜는 것이 다음 실험 1순위다.

## 1. 설정 (`CFG`)

모든 설정을 한곳에 모았다. 환경변수로 덮어쓸 수 있다.
예) PowerShell: `$env:EXP="lr_c03"; $env:LR_C="0.3"; python v10_clean.py`

In [ ]:
import os
import json
import time
import hashlib
import warnings
from collections import Counter, namedtuple
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)


def env(key, default):
    """환경변수 key가 있으면 default와 같은 타입으로 바꿔서 돌려준다.
    주의: bool("False")는 True라서, bool은 "1/true/yes"만 참으로 따로 처리한다."""
    v = os.environ.get(key)
    if v is None:
        return default
    if isinstance(default, bool):
        return v.strip().lower() in ("1", "true", "yes")
    return type(default)(v)


CFG = {k: env(k, d) for k, d in dict(
    EXP="baseline",        # 실험 이름. 결과는 out/<EXP>/ 에 저장
    DATA_DIR=".",          # train.csv 등이 있는 폴더. 팀 레포 models/민준/ 에서는 "../../data"
    SEED=42,               # 모든 난수 고정 (팀 공통)
    N_FOLDS=5,             # CV fold 수 (팀 공통 folds.csv와 같아야 함)
    HGVS_LEGACY=False,     # True면 'R1450X'의 X를 종결 코돈(*)으로 읽음. 운영진 답 전까지 False
    MIN_GENE=5,            # g_<유전자>: 학습 부분에서 변이 보유자가 이 수 이상인 유전자만 사용
    MIN_TRUNC=3,           # t_<유전자>: 절단형(nonsense/frameshift) 보유자가 이 수 이상
    MIN_HS=3,              # hs_<유전자>|<위치>: 그 위치 변이 보유자가 이 수 이상
    LR_C=0.1,              # 로지스틱 회귀 규제 세기 (작을수록 강한 규제). 민정 님 v3 값
    NUM_LEAVES=15,         # LightGBM 트리 잎 개수
    LGB_LR=0.05,           # LightGBM 학습률
    LGB_COLSAMPLE=0.3,     # 트리마다 피처 30%만 사용 (희소 피처가 많아 과적합 방지)
    LGB_MAX_ROUNDS=3000,   # 최대 학습 횟수 (early stopping으로 보통 훨씬 일찍 멈춤)
    LGB_PATIENCE=100,      # 내부 검증 손실이 100번 연속 개선되지 않으면 중단
    INNER_VALID=0.1,       # early stopping용 내부 검증 비율 (fold의 학습 부분을 다시 쪼갬)
    USE_AA=False,          # 아미노산 치환쌍 피처 (수요일 실험용)
    MIN_AA=20,             # 치환쌍 사용 기준 (학습 부분 합계)
    USE_XGB=False,         # XGBoost 추가 (수요일 실험용)
    USE_SIG=True,          # 변이 시그니처: 아미노산 변화로 추정한 DNA 염기 치환 6종 비율 (4절)
    USE_TWIN=True,         # 쌍둥이 규칙: test 행이 train의 KIRC/KIPAN/LGG/GBMLGG 환자와 프로필이 똑같으면 짝 암종으로 (8-1절)
    USE_GF=True,           # Geneformer(사전학습 모델) 유전자 임베딩 피처. 모델 가중치만 쓰고 외부 데이터는 안 씀
    GF_DIR="pretrained/geneformer_v2",   # Hugging Face ctheodoris/Geneformer 파일 폴더. V2-104M(채택). V1-10M은 pretrained/geneformer
    BASE_EXP="",           # 비교 기준 실험 이름. 있으면 채택 규칙(adopt) 결과를 출력
    TEAM_DIR="team",       # 팀원 확률 파일(oof_<이름>.npy, test_<이름>.npy) 폴더
).items()}
print(json.dumps(CFG, ensure_ascii=False, indent=1))

In [ ]:
os.environ["_T_BOOL"] = "False"; assert env("_T_BOOL", True) is False
os.environ["_T_BOOL"] = "1";     assert env("_T_BOOL", False) is True
os.environ["_T_INT"] = "31";     assert env("_T_INT", 15) == 31
assert env("_T_MISSING", 0.1) == 0.1

## 2. 데이터 불러오기

- 셀 값: 변이가 없으면 `WT`, 있으면 단백질 변이 표기(예: `R175H`). 빈 칸(NaN)은 `WT`로 본다.
- 클래스(열) 순서는 `LabelEncoder` 기본값인 **알파벳순**. 팀원 확률 파일도 이 순서를 따른다.

In [ ]:
from sklearn.preprocessing import LabelEncoder

D = Path(CFG["DATA_DIR"])
t0 = time.time()
train = pd.read_csv(D / "train.csv").fillna("WT")
test = pd.read_csv(D / "test.csv").fillna("WT")
sample = pd.read_csv(D / "sample_submission.csv")
genes = list(train.columns[2:])          # ID, SUBCLASS 다음부터 유전자 4,384개

assert train.shape == (6201, 4386), train.shape
assert test.shape == (2546, 4385), test.shape
assert list(test.columns[1:]) == genes, "train/test 유전자 열 이름·순서가 다릅니다"
assert (test.ID.values == sample.ID.values).all(), "test와 sample_submission의 ID 순서가 다릅니다"

le = LabelEncoder()
y = le.fit_transform(train.SUBCLASS)
CLASSES = le.classes_
assert len(CLASSES) == 26

OUT = Path("out") / CFG["EXP"]
OUT.mkdir(parents=True, exist_ok=True)
print(f"train {train.shape}, test {test.shape}, 클래스 {len(CLASSES)}개, 불러오기 {time.time() - t0:.1f}s")

## 3. 파서: 셀 글자 → 변이 정보

셀 하나를 공백으로 나누고, 토큰마다 종류를 정한다. **한 행씩 독립적으로** 처리한다(다른 행을 보지 않음).

| 종류 | 예시 | 판단 규칙 |
|---|---|---|
| missense (아미노산이 바뀜) | `R175H` | 아미노산 + 위치 + 다른 아미노산 |
| silent (바뀌지 않음) | `L858L` | 앞뒤 아미노산이 같음 |
| nonsense (단백질이 중간에 끊김) | `R1450*` | 끝이 `*` |
| frameshift (읽는 틀이 밀림) | `E1554fs`, `S221Kfs` | `fs`가 들어 있음 |
| inframe (일부 삭제·삽입) | `E746_A750del` | `del`/`ins`/`dup`가 들어 있음 |
| other | 그 외 | |

`HGVS_LEGACY=False`(기본)이면 `R1450X`의 `X`는 20개 표준 아미노산이 아니므로 `other`로 둔다.

In [ ]:
import re

KINDS = ["missense", "silent", "nonsense", "frameshift", "inframe", "other"]
AA20 = set("ACDEFGHIKLMNPQRSTVWY")            # 20개 표준 아미노산
RE_SUB = re.compile(r"^([A-Z*])(\d+)([A-Z*])$")  # 한 글자 치환: R175H, R1450*, L858L
RE_NUM = re.compile(r"\d+")


def parse_token(tok, legacy=False):
    """변이 토큰 1개 → (종류, 원래 아미노산, 위치, 바뀐 아미노산). 없는 값은 "" / 위치 -1.
    다른 토큰·다른 행을 전혀 보지 않는다 (행 독립)."""
    t = tok.strip()
    if t.startswith("p."):
        t = t[2:]
    m = RE_SUB.match(t)
    if m and "fs" not in t.lower():
        ref, pos, alt = m.group(1), int(m.group(2)), m.group(3)
        if alt == "*":
            kind = "nonsense"
        elif alt == "X":                        # 옛 HGVS 표기의 종결 코돈. 운영진 답 전까지 기본은 other
            kind = "nonsense" if legacy else "other"
        elif ref == alt:
            kind = "silent"
        elif ref in AA20 and alt in AA20:
            kind = "missense"
        else:
            kind = "other"
        return kind, ref, pos, alt
    low = t.lower()
    if "fs" in low:
        kind = "frameshift"
    elif "del" in low or "ins" in low or "dup" in low:
        kind = "inframe"
    else:
        kind = "other"
    n = RE_NUM.search(t)
    ref = t[0] if t and t[0] in AA20 else ""
    return kind, ref, (int(n.group()) if n else -1), ""


def parse_rows(df, genes, legacy=False):
    """DataFrame → 행마다 [(유전자 번호, 종류, 원래 aa, 위치, 바뀐 aa), ...].
    WT·빈 칸은 건너뛴다. 각 행은 그 행의 셀만 보고 만든다."""
    vals = df[genes].to_numpy(dtype=object)
    rows = [[] for _ in range(len(df))]
    for r, c in zip(*np.nonzero(vals != "WT")):
        for tok in str(vals[r, c]).split():
            if tok != "WT":
                k, ref, pos, alt = parse_token(tok, legacy)
                rows[r].append((int(c), k, ref, pos, alt))
    return rows

In [ ]:
P = parse_token
assert P("R175H")[:3] == ("missense", "R", 175)
assert P("L858L")[0] == "silent"
assert P("R1450*")[0] == "nonsense"
assert P("E1554fs")[:3] == ("frameshift", "E", 1554)
assert P("S221Kfs")[:3] == ("frameshift", "S", 221)
assert P("E746_A750del")[:3] == ("inframe", "E", 746)
assert P("R1450X")[0] == "other"                    # HGVS_LEGACY 꺼짐
assert P("R1450X", legacy=True)[0] == "nonsense"    # 켜짐


def test_parse_odd_tokens():                         # 예상 밖 표기도 오류 없이 분류
    for t in ["1529_1530AL>AL", "*123fs", "X123Y", "abc", "123"]:
        assert P(t)[0] in KINDS, t
    assert P("p.R175H")[:3] == ("missense", "R", 175)


test_parse_odd_tokens()


def test_parse_weird_cells():                        # 빈 칸, 앞뒤 공백, 공백 여러 칸
    df = pd.DataFrame({"ID": ["a", "b", "c"], "G1": ["WT", "  R175H   V600E ", ""], "G2": ["WT", "WT", "WT"]})
    r = parse_rows(df, ["G1", "G2"])
    assert r[0] == [] and r[2] == []
    assert [e[1] for e in r[1]] == ["missense", "missense"]


test_parse_weird_cells()

In [ ]:
t0 = time.time()
rows_tr = parse_rows(train, genes, CFG["HGVS_LEGACY"])
rows_te = parse_rows(test, genes, CFG["HGVS_LEGACY"])
print(f"파싱 {time.time() - t0:.1f}s")
print("train 토큰 종류별 개수:", dict(Counter(e[1] for r in rows_tr for e in r)))

## 4. 피처 만들기 (`FeatureBuilder`)

- `fit(rows, ids)`: **학습 부분의 행만** 보고 "어떤 유전자·위치를 피처로 쓸지"(단어장)를 정한다.
- `transform(rows)`: 정해진 단어장대로 숫자 행렬로 바꾼다. 단어장에 없는 변이(예: test에만 있는 변이)는 무시된다.

| 피처 묶음 | 이름 예시 | 포함 기준 (학습 부분에서 셈) | 의미 |
|---|---|---|---|
| 유전자 변이 여부 | `g_TP53` | 보유자 `MIN_GENE`(5)명 이상 | 이 유전자가 고장났나? |
| 절단형 변이 여부 | `t_APC` | 절단형 보유자 `MIN_TRUNC`(3)명 이상 | 유전자가 끊어졌나? (APC→대장암, VHL→신장암) |
| 위치 핫스팟 | `hs_BRAF\|V600` | 보유자 `MIN_HS`(3)명 이상 | 정확히 이 위치가 고장났나? (바뀐 아미노산 표기와 무관) |
| 변이량 | `log_n_genes`, `n_events`, `cnt_*`, `frac_*` | 항상 | 변이가 적은 암(PRAD·THCA) vs 많은 암(SKCM·UCEC) |
| 변이 시그니처 (`USE_SIG`) | `sig_C>T` 등 6개 | 항상 (유전 암호표로 계산) | 발암 원인의 흔적: 자외선(SKCM) C>T, 흡연(LUSC·LUAD) C>A, APOBEC(CESC·BLCA) |
| Geneformer (`USE_GF`, LR 전용) | `gf_mean_*`, `gf_max_*` | 항상 (사전학습 표) | 유전자가 어떤 조직·세포에서 쓰이는지 (4-1절) |

"보유자 수"는 **행(환자) 수**다. 한 환자에게 여러 번 나와도 1명으로 센다.
변이량은 지우면 안 된다: V6에서 지웠다가 애매한 환자를 전부 PRAD로 찍는 버그가 났다.

In [ ]:
import scipy.sparse as sp

TRUNC = {"nonsense", "frameshift"}       # 절단형 변이
NO_HS = {"silent", "other"}              # 핫스팟에서 제외 (기능 변화가 없거나 해석 불가)

# 변이 시그니처 (도메인 지식): 발암 원인마다 남기는 DNA 치환 흔적이 다르다.
#   자외선(SKCM) → C>T, 흡연(LUAD·LUSC) → C>A, APOBEC(BLCA·CESC) → C>T·C>G
# 데이터에는 DNA가 없고 아미노산 변화만 있으므로, 표준 유전 암호표(64개 코돈)로
# "이 아미노산 변화를 만들 수 있는 염기 하나 치환"을 모두 찾아 같은 비중으로 나눠 센다.
# 치환은 관례대로 피리미딘(C·T) 기준 6종으로 접는다 (G>A는 C>T와 같은 것으로 셈).
SBS6 = ["C>A", "C>G", "C>T", "T>A", "T>C", "T>G"]
_B = "TCAG"
CODON = {a + b + c: aa for (a, b, c), aa in zip(((a, b, c) for a in _B for b in _B for c in _B),
                                                "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG")}
_COMP = {"A": "T", "C": "G", "G": "C", "T": "A"}


def _sig_table():
    """(바뀌기 전 아미노산, 바뀐 후) → SBS6 비중 벡터(합 1). 염기 하나로 만들 수 없는 변화는 없음."""
    acc = {}
    for cod, a in CODON.items():
        for i in range(3):
            for x in "ACGT":
                if x == cod[i]:
                    continue
                b = CODON[cod[:i] + x + cod[i + 1:]]
                r, t = cod[i], x
                if r in "AG":                       # 퓨린 기준이면 상보 가닥(피리미딘 기준)으로 접기
                    r, t = _COMP[r], _COMP[t]
                acc.setdefault((a, b), np.zeros(6))[SBS6.index(f"{r}>{t}")] += 1
    return {k: v / v.sum() for k, v in acc.items()}


SIG = _sig_table()


class FeatureBuilder:
    def __init__(self, genes, min_gene=5, min_trunc=3, min_hs=3, use_aa=False, min_aa=20, gf_emb=None, use_sig=False):
        self.genes = genes
        self.min_gene, self.min_trunc, self.min_hs = min_gene, min_trunc, min_hs
        self.use_aa, self.min_aa = use_aa, min_aa
        self.use_sig = use_sig
        self.gf_emb = gf_emb      # (유전자 수, d) 사전학습 임베딩 표. 고정값이라 fit에서 배울 것이 없음

    @staticmethod
    def _keys(row):
        """한 행 → (변이 유전자 집합, 절단형 유전자 집합, 핫스팟 키 집합)"""
        g = {e[0] for e in row}
        t = {e[0] for e in row if e[1] in TRUNC}
        h = {(e[0], f"{e[2]}{e[3]}") for e in row if e[1] not in NO_HS and e[3] >= 0}
        return g, t, h

    def fit(self, rows, ids):
        """학습 부분의 행만으로 단어장을 정한다. ids는 누수 검사용으로 기록만 한다."""
        self.fit_ids = np.asarray(ids)
        cg, ct, ch, caa = Counter(), Counter(), Counter(), Counter()
        for row in rows:
            g, t, h = self._keys(row)
            cg.update(g); ct.update(t); ch.update(h)
            if self.use_aa:
                caa.update((e[2], e[4]) for e in row if e[1] == "missense")
        self.g_idx = {k: i for i, k in enumerate(sorted(k for k, v in cg.items() if v >= self.min_gene))}
        self.t_idx = {k: i for i, k in enumerate(sorted(k for k, v in ct.items() if v >= self.min_trunc))}
        self.h_idx = {k: i for i, k in enumerate(sorted(k for k, v in ch.items() if v >= self.min_hs))}
        self.aa_idx = {k: i for i, k in enumerate(sorted(k for k, v in caa.items() if v >= self.min_aa))}
        self.burden_names = ["log_n_genes", "n_events"] + [f"cnt_{k}" for k in KINDS] + [f"frac_{k}" for k in KINDS]
        G = self.genes
        self.names = ([f"g_{G[k]}" for k in self.g_idx] + [f"t_{G[k]}" for k in self.t_idx] +
                      [f"hs_{G[k[0]]}|{k[1]}" for k in self.h_idx] + self.burden_names +
                      [f"aa_{a}>{b}" for a, b in self.aa_idx] +
                      ([f"sig_{k}" for k in SBS6] if self.use_sig else []))
        self.n_lgb = len(self.names)   # LightGBM은 앞의 이 열들만 쓴다 (GF 열은 LR 전용)
        if self.gf_emb is not None:
            d = self.gf_emb.shape[1]
            self.names += [f"gf_mean_{i}" for i in range(d)] + [f"gf_max_{i}" for i in range(d)]
        return self

    def transform(self, rows):
        """단어장대로 희소 행렬(float32)로 바꾼다. 각 행은 그 행만 보고 변환된다."""
        o_t = len(self.g_idx); o_h = o_t + len(self.t_idx); o_b = o_h + len(self.h_idx)
        o_a = o_b + len(self.burden_names)
        R, C, V = [], [], []
        for r, row in enumerate(rows):
            g, t, h = self._keys(row)
            cols = ([self.g_idx[k] for k in g if k in self.g_idx] +
                    [o_t + self.t_idx[k] for k in t if k in self.t_idx] +
                    [o_h + self.h_idx[k] for k in h if k in self.h_idx])
            R += [r] * len(cols); C += cols; V += [1.0] * len(cols)
            cnt = Counter(e[1] for e in row); n_ev = len(row)
            b = [np.log1p(len(g)), n_ev] + [cnt[k] for k in KINDS] + [cnt[k] / (n_ev + 1) for k in KINDS]
            R += [r] * len(b); C += list(range(o_b, o_b + len(b))); V += b
            if self.aa_idx:
                ca = Counter((e[2], e[4]) for e in row if e[1] == "missense")
                for k, v in ca.items():
                    if k in self.aa_idx:
                        R.append(r); C.append(o_a + self.aa_idx[k]); V.append(v / (cnt["missense"] + 1))
            if self.use_sig:
                o_s = o_a + len(self.aa_idx)
                sv = sum((SIG[(e[2], "*" if e[4] == "X" else e[4])] for e in row
                          if e[1] in ("missense", "nonsense", "silent") and (e[2], "*" if e[4] == "X" else e[4]) in SIG),
                         np.zeros(6))
                if sv.sum() > 0:
                    sv = sv / sv.sum()
                R += [r] * 6; C += list(range(o_s, o_s + 6)); V += sv.tolist()
            if self.gf_emb is not None:
                gi = sorted({e[0] for e in row if e[1] != "silent"})
                if gi:          # 변이가 없으면 0으로 둔다
                    M = self.gf_emb[gi]
                    b = np.concatenate([M.mean(0), M.max(0)])
                    o_g = len(self.names) - len(b)
                    R += [r] * len(b); C += list(range(o_g, o_g + len(b))); V += b.tolist()
        X = sp.csr_matrix((np.asarray(V, np.float32), (R, C)), shape=(len(rows), len(self.names)))
        X.eliminate_zeros()
        return X

### 4-1. Geneformer 유전자 임베딩 (`USE_GF`)

- [Geneformer](https://huggingface.co/ctheodoris/Geneformer)(V1-10M, Apache-2.0)는 단일세포 약 3천만 개의 유전자 발현으로 **미리 학습된 모델**이다.
  유전자마다 "어떤 세포·조직에서 함께 쓰이는지"를 담은 256차원 벡터(입력 임베딩 표)가 있다.
- 우리는 모델을 실행하지 않고 **이 표만 꺼내 쓴다**. 유전자 기호 → Ensembl ID → 토큰 번호 → 벡터 순으로 찾는다.
- 환자 피처: silent를 뺀 변이 유전자들의 벡터 **평균**(`gf_mean_*`)과 **최댓값**(`gf_max_*`). 사전에 없는 유전자는 0 벡터.
- **로지스틱 회귀에만** 넣는다. 둘 다 넣었을 때(실험 `gf`) LR은 +0.006 올랐지만 LGB가 −0.030 떨어져 전체 0.4172로 기준보다 낮았다.
- 누수 없음: 표는 대회 데이터와 무관하게 고정된 값이고, 각 행은 자기 변이만 보고 계산된다(fit에서 배우는 것 없음).
- 보안: `.bin`은 `torch.load(weights_only=True)`, `.pkl`은 numpy 숫자 외의 객체를 막은 Unpickler로 읽는다.
- 필요한 파일 (`GF_DIR` 안): V2-104M(기본, 채택) = `Geneformer-V2-104M/model.safetensors` + `geneformer/token_dictionary_gc104M.pkl` + `geneformer/gene_name_id_dict_gc104M.pkl` / V1-10M = `Geneformer-V1-10M/pytorch_model.bin` + `gene_dictionaries_30m/` 의 gc30M 사전 2개
- 결과: V1-10M(256차원) CV 0.4343, V2-104M(768차원) CV **0.4441** (기준 0.4243)

In [ ]:
import pickle


class _SafeUnpickler(pickle.Unpickler):
    """사전(dict) 파일 전용: numpy 숫자 복원만 허용하고 그 밖의 코드 실행은 막는다."""
    SAFE = {("numpy.core.multiarray", "scalar"): np._core.multiarray.scalar, ("numpy", "dtype"): np.dtype}

    def find_class(self, module, name):
        if (module, name) in self.SAFE:
            return self.SAFE[(module, name)]
        raise pickle.UnpicklingError(f"허용되지 않은 객체: {module}.{name}")


def read_safetensor(path, key):
    """safetensors 파일에서 F32 텐서 하나만 읽는다 (라이브러리 없이: 8바이트 길이 + JSON 머리말 + 원본 바이트).
    숫자 배열만 담는 형식이라 코드가 실행될 위험이 없다."""
    with open(path, "rb") as f:
        n = int.from_bytes(f.read(8), "little")
        meta = json.loads(f.read(n))[key]
        assert meta["dtype"] == "F32", meta["dtype"]
        a, b = meta["data_offsets"]
        f.seek(8 + n + a)
        return np.frombuffer(f.read(b - a), np.float32).reshape(meta["shape"]).copy()


def load_gf_table(gf_dir, genes):
    """우리 유전자 순서대로 (유전자 수, d) 임베딩 표. 사전에 없는 유전자는 0 행.
    V1-10M(pytorch_model.bin, d=256)과 V2-104M(model.safetensors, d=768) 모두 읽는다."""
    d = Path(gf_dir)
    tok = _SafeUnpickler(open(next(d.glob("token_dictionary_*.pkl")), "rb")).load()        # Ensembl ID → 토큰 번호
    name2id = _SafeUnpickler(open(next(d.glob("gene_name_id_dict_*.pkl")), "rb")).load()  # 유전자 기호 → Ensembl ID
    key = "bert.embeddings.word_embeddings.weight"
    if (d / "model.safetensors").exists():
        W = read_safetensor(d / "model.safetensors", key)
    else:
        import torch
        W = torch.load(d / "pytorch_model.bin", weights_only=True, map_location="cpu")[key].numpy().astype(np.float32)
    T = np.zeros((len(genes), W.shape[1]), np.float32)
    hit = 0
    for i, g in enumerate(genes):
        if g in name2id and name2id[g] in tok:
            T[i] = W[int(tok[name2id[g]])]; hit += 1
    print(f"Geneformer 임베딩: 유전자 {len(genes)}개 중 {hit}개 연결 ({hit / len(genes):.1%})")
    return T


GF_EMB = load_gf_table(CFG["GF_DIR"], genes) if CFG["USE_GF"] else None
if GF_EMB is not None:
    assert GF_EMB.shape[0] == len(genes) and np.isfinite(GF_EMB).all()
    assert (np.abs(GF_EMB).sum(1) > 0).mean() > 0.95, "연결률이 너무 낮음: 파일이나 유전자 이름을 확인"

In [ ]:
import struct, tempfile
_W = np.arange(6, dtype=np.float32).reshape(3, 2)
_hdr = json.dumps({"other": {"dtype": "F32", "shape": [1], "data_offsets": [0, 4]},
                   "emb": {"dtype": "F32", "shape": [3, 2], "data_offsets": [4, 28]}}).encode()
with tempfile.TemporaryDirectory() as _d:
    _p = Path(_d) / "t.safetensors"
    _p.write_bytes(struct.pack("<Q", len(_hdr)) + _hdr + np.float32([9]).tobytes() + _W.tobytes())
    assert (read_safetensor(_p, "emb") == _W).all()

In [ ]:
G = ["A", "B", "C"]
mk = lambda *ev: list(ev)
toy = [mk((0, "missense", "R", 10, "H"))] * 5 + [mk((1, "nonsense", "Q", 5, "*"))] * 4 + [mk()]
fb = FeatureBuilder(G, min_gene=5, min_trunc=3, min_hs=3).fit(toy, np.arange(len(toy)))
assert "g_A" in fb.names and "g_B" not in fb.names          # B 보유자 4명 < 5
assert "t_B" in fb.names                                    # 절단형 보유자 4명 ≥ 3
assert "hs_A|R10" in fb.names
assert {"log_n_genes", "n_events"} <= set(fb.names)
assert all(f"cnt_{k}" in fb.names and f"frac_{k}" in fb.names for k in KINDS)
X = fb.transform(toy)
assert X.shape == (len(toy), len(fb.names))
unseen = fb.transform([mk((2, "missense", "K", 1, "E"))])   # 학습 때 없던 유전자 C
assert unseen[0, fb.names.index("g_A")] == 0
# 행 독립: 한꺼번에 변환 == 한 행씩 변환
assert (X.toarray() == np.vstack([fb.transform([r]).toarray() for r in toy])).all()

# Geneformer 임베딩 피처: silent를 뺀 변이 유전자 벡터의 평균·최댓값
E = np.array([[1, 0], [3, 2], [5, -4]], np.float32)            # 유전자 A, B, C의 2차원 가짜 임베딩
fb_gf = FeatureBuilder(G, min_gene=5, min_trunc=3, min_hs=3, gf_emb=E).fit(toy, np.arange(len(toy)))
assert {"gf_mean_0", "gf_mean_1", "gf_max_0", "gf_max_1"} <= set(fb_gf.names)
row = mk((0, "missense", "R", 10, "H"), (1, "nonsense", "Q", 5, "*"), (2, "silent", "K", 1, "K"))
v = fb_gf.transform([row]).toarray()[0]
col = lambda n: v[fb_gf.names.index(n)]
assert col("gf_mean_0") == 2 and col("gf_mean_1") == 1          # (A+B)/2, C는 silent라 제외
assert col("gf_max_0") == 3 and col("gf_max_1") == 2
assert not fb_gf.transform([mk()]).toarray()[0, -4:].any()      # 변이 없는 환자 → 0
assert fb.names == fb_gf.names[:len(fb.names)]                  # 기존 피처는 그대로
assert fb_gf.n_lgb == len(fb.names) and fb.n_lgb == len(fb.names)   # LGB는 GF 열을 보지 않음 (LR 전용)

# 변이 시그니처: 아미노산 변화 → 가능한 DNA 염기 치환(6종) 비율
fb_sig = FeatureBuilder(G, min_gene=5, min_trunc=3, min_hs=3, use_sig=True).fit(toy, np.arange(len(toy)))
assert all(f"sig_{k}" in fb_sig.names for k in SBS6)
row = mk((0, "missense", "R", 175, "H"), (1, "missense", "V", 600, "E"))   # R>H는 C>T뿐, V>E는 T>A뿐
v = fb_sig.transform([row]).toarray()[0]
assert v[fb_sig.names.index("sig_C>T")] == 0.5 and v[fb_sig.names.index("sig_T>A")] == 0.5
assert np.isclose(sum(v[fb_sig.names.index(f"sig_{k}")] for k in SBS6), 1)
assert fb_sig.n_lgb == len(fb_sig.names)                                    # 시그니처는 LGB·LR 모두 사용

## 5. Fold 나누기 (`folds.csv`, 팀 공통)

- train을 5개 묶음으로 나눠 "4개로 학습, 1개로 채점"을 5번 반복한다.
- `StratifiedGroupKFold`: 묶음마다 암종 비율을 비슷하게(Stratified) + **변이 프로필이 완전히 같은 환자는 같은 묶음에**(Group).
  같은 프로필이 학습과 채점에 나뉘면 정답을 외워 맞히는 셈이라 점수가 부풀려진다.
- `folds.csv`가 이미 있으면 그대로 읽는다(팀 공통 파일이므로 덮어쓰지 않는다).

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold


def profile_hashes(df, genes):
    """행마다 변이 프로필(유전자 셀 4,384개)의 지문(md5 앞 12자리). 셀이 전부 같으면 지문이 같다."""
    return np.array([hashlib.md5("|".join(r).encode()).hexdigest()[:12] for r in df[genes].to_numpy(dtype=object)])


def make_folds(train, genes, y, n_splits, seed):
    """같은 변이 프로필(유전자 셀 4,384개가 전부 같음) = 같은 그룹. 그룹 단위로 층화 분할."""
    vals = train[genes].to_numpy(dtype=object)
    group = profile_hashes(train, genes)
    fold = np.full(len(train), -1)
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for k, (_, va) in enumerate(sgkf.split(vals, y, group)):
        fold[va] = k
    return pd.DataFrame({"ID": train.ID.values, "fold": fold, "group": group})


def check_folds(folds, y, n_classes):
    """fold 무결성 검사. 하나라도 어기면 AssertionError."""
    assert folds.ID.is_unique, "ID 중복"
    assert set(folds.fold) == set(range(folds.fold.max() + 1)), "fold 번호가 0..K-1이 아님"
    assert (folds.groupby("group").fold.nunique() == 1).all(), "같은 그룹이 여러 fold에 나뉨"
    for k in sorted(set(folds.fold)):
        assert len(set(y[folds.fold.values != k])) == n_classes, f"fold {k}의 학습 부분에 없는 클래스가 있음"


FOLD_PATH = Path("folds.csv")
if FOLD_PATH.exists():
    folds = pd.read_csv(FOLD_PATH).set_index("ID").loc[train.ID].reset_index()
    print("folds.csv 불러옴")
else:
    folds = make_folds(train, genes, y, CFG["N_FOLDS"], CFG["SEED"])
    folds.to_csv(FOLD_PATH, index=False)
    print("folds.csv 새로 만듦")
fold_arr = folds.fold.values
gs = folds.groupby("group").size(); nl = pd.Series(train.SUBCLASS.values).groupby(folds.group.values).nunique()
print(f"중복 프로필 그룹 {(gs > 1).sum()}개 (샘플 {gs[gs > 1].sum()}명), 그중 라벨이 서로 다른 그룹 {(nl > 1).sum()}개")
print("fold별 크기:", folds.fold.value_counts().sort_index().to_dict())

In [ ]:
check_folds(folds, y, 26)
assert len(folds) == 6201 and (folds.ID.values == train.ID.values).all()
# 일부러 같은 그룹을 두 fold에 나눠 놓으면 check_folds가 잡아내야 한다
bad = folds.copy()
dup_groups = folds.group.value_counts(); g = dup_groups[dup_groups > 1].index[0]
bad.loc[bad.group == g, "fold"] = [0, 1] + [0] * ((bad.group == g).sum() - 2)
try:
    check_folds(bad, y, 26)
    raise RuntimeError("그룹이 나뉜 fold를 통과시키면 안 됨")
except AssertionError:
    pass

## 6. 학습 (5-fold CV): LightGBM + 로지스틱 회귀

fold k마다:
1. 학습 부분으로만 `FeatureBuilder.fit` → 학습/검증/test 변환
2. **LightGBM**: 학습 부분을 다시 90/10으로 쪼개 10%로만 early stopping (검증 fold는 보지 않음)
3. **로지스틱 회귀**: 학습 부분 전체로 학습
4. 두 확률을 **0.5 : 0.5 고정**으로 평균 (OOF를 보며 비율을 찾으면 점수가 부풀려짐)
5. 검증 부분 예측 → OOF, test 예측 → 5개 fold 평균

채택 규칙(`adopt`): 새 실험은 기준 실험보다 **전체 OOF가 +0.003 이상** 오르고 **5개 fold 중 3개 이상**에서 올라야 채택한다.

In [ ]:
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MaxAbsScaler

warnings.filterwarnings("ignore", message=".*eval_set.*deprecated.*")   # lightgbm 4.7 안내 문구

FoldModel = namedtuple("FoldModel", "fb lgb lr")


def macro_f1(y_true, p):
    """확률 → argmax → Macro F1 (26개 암종 F1의 단순 평균)"""
    return f1_score(y_true, p.argmax(1), average="macro")


def fit_fold(rows, ids, y_fold, cfg):
    """학습 부분 하나로 피처 기준 + LGB + LR을 학습한다. 검증 fold·test는 여기 들어오지 않는다."""
    fb = FeatureBuilder(genes, cfg["MIN_GENE"], cfg["MIN_TRUNC"], cfg["MIN_HS"],
                        cfg["USE_AA"], cfg["MIN_AA"], GF_EMB, use_sig=cfg["USE_SIG"]).fit(rows, ids)
    X = fb.transform(rows)
    # early stopping용 내부 검증: 학습 부분을 다시 90/10으로 (검증 fold는 절대 보지 않음)
    i_tr, i_va = train_test_split(np.arange(len(rows)), test_size=cfg["INNER_VALID"],
                                  stratify=y_fold, random_state=cfg["SEED"])
    m_lgb = lgb.LGBMClassifier(objective="multiclass", class_weight="balanced",
                               num_leaves=cfg["NUM_LEAVES"], learning_rate=cfg["LGB_LR"],
                               colsample_bytree=cfg["LGB_COLSAMPLE"], n_estimators=cfg["LGB_MAX_ROUNDS"],
                               random_state=cfg["SEED"], verbose=-1)
    XL = X[:, :fb.n_lgb]       # GF 열 제외: 촘촘한 GF 512열이 LGB의 피처 샘플링을 흐려 점수가 떨어졌음 (gf 실험)
    m_lgb.fit(XL[i_tr], y_fold[i_tr], eval_set=[(XL[i_va], y_fold[i_va])],
              callbacks=[lgb.early_stopping(cfg["LGB_PATIENCE"], verbose=False)])
    m_lr = make_pipeline(MaxAbsScaler(), LogisticRegression(C=cfg["LR_C"], class_weight="balanced", max_iter=1000))
    m_lr.fit(X, y_fold)
    return FoldModel(fb, m_lgb, m_lr)


def predict_parts(m, rows):
    """(LGB 확률, LR 확률). 각 행은 독립적으로 변환·예측된다."""
    X = m.fb.transform(rows)
    return m.lgb.predict_proba(X[:, :m.fb.n_lgb]), m.lr.predict_proba(X)


def predict_fold(m, rows):
    """LGB와 LR을 0.5 : 0.5로 고정해서 섞은 확률 (n, 26)"""
    a, b = predict_parts(m, rows)
    return 0.5 * a + 0.5 * b


def adopt(new_oof, base_oof, y_true, fold_arr):
    """채택 규칙: 전체 Macro F1이 +0.003 이상, 그리고 5개 fold 중 3개 이상에서 상승"""
    gain = macro_f1(y_true, new_oof) - macro_f1(y_true, base_oof)
    wins = sum(macro_f1(y_true[fold_arr == k], new_oof[fold_arr == k]) >
               macro_f1(y_true[fold_arr == k], base_oof[fold_arr == k]) for k in sorted(set(fold_arr)))
    return bool(gain >= 0.003 and wins >= 3)


def run_cv(rows_tr, rows_te, y, fold_arr, cfg):
    K = cfg["N_FOLDS"]; n_cls = len(CLASSES)
    oof_lgb = np.zeros((len(y), n_cls)); oof_lr = np.zeros((len(y), n_cls))
    te = np.zeros((len(rows_te), n_cls)); models, iters = [], []
    for k in range(K):
        t0 = time.time()
        trn, val = np.where(fold_arr != k)[0], np.where(fold_arr == k)[0]
        m = fit_fold([rows_tr[i] for i in trn], trn, y[trn], cfg)
        oof_lgb[val], oof_lr[val] = predict_parts(m, [rows_tr[i] for i in val])
        te += predict_fold(m, rows_te) / K
        models.append(m); iters.append(int(m.lgb.best_iteration_))
        f = lambda p: macro_f1(y[val], p[val])
        print(f"fold{k} | 피처 {len(m.fb.names)} | LGB it={iters[-1]} F1={f(oof_lgb):.4f} | "
              f"LR F1={f(oof_lr):.4f} | 섞음 F1={f(0.5 * oof_lgb + 0.5 * oof_lr):.4f} | {time.time() - t0:.0f}s")
    oof = 0.5 * oof_lgb + 0.5 * oof_lr
    fold_f1 = [macro_f1(y[fold_arr == k], oof[fold_arr == k]) for k in range(K)]
    return dict(oof=oof, te=te, oof_lgb=oof_lgb, oof_lr=oof_lr, models=models, fold_f1=fold_f1,
                cv_f1=macro_f1(y, oof), lgb_f1=macro_f1(y, oof_lgb), lr_f1=macro_f1(y, oof_lr), best_iters=iters)


res = run_cv(rows_tr, rows_te, y, fold_arr, CFG)
print(f"\n[{CFG['EXP']}] CV Macro F1 = {res['cv_f1']:.4f} (LGB {res['lgb_f1']:.4f}, LR {res['lr_f1']:.4f}) "
      f"| fold별 {np.round(res['fold_f1'], 4).tolist()} ± {np.std(res['fold_f1']):.4f}")

# 결과 저장: out/<EXP>/ + out/summary.csv 한 줄
np.save(OUT / "oof.npy", res["oof"]); np.save(OUT / "test.npy", res["te"])
json.dump({"cfg": CFG, **{k: res[k] for k in ["cv_f1", "lgb_f1", "lr_f1", "fold_f1", "best_iters"]}},
          open(OUT / "cv.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
pd.DataFrame([{"exp": CFG["EXP"], "cv_f1": round(res["cv_f1"], 5), "fold_f1": np.round(res["fold_f1"], 4).tolist(),
               "cfg": json.dumps(CFG, ensure_ascii=False)}]).to_csv(
    "out/summary.csv", mode="a", header=not Path("out/summary.csv").exists(), index=False, encoding="utf-8-sig")
if CFG["BASE_EXP"]:
    base = np.load(Path("out") / CFG["BASE_EXP"] / "oof.npy")
    print(f"기준 [{CFG['BASE_EXP']}] {macro_f1(y, base):.4f} 대비 채택(adopt): {adopt(res['oof'], base, y, fold_arr)}")

In [ ]:
assert res["oof"].shape == (6201, 26) and res["te"].shape == (2546, 26)
assert np.allclose(res["oof"].sum(1), 1, atol=1e-5) and np.allclose(res["te"].sum(1), 1, atol=1e-5)
assert len(res["fold_f1"]) == 5 and len(res["models"]) == 5
assert res["cv_f1"] > 0.30, "기본 모델이 너무 낮음 — 파이프라인 오류 의심 (목표치 아님)"
assert adopt(res["oof"], res["oof"], y, fold_arr) is False          # 같은 결과는 채택 안 됨

## 7. 누수 자동 검사 (하나라도 실패하면 여기서 멈춘다)

| # | 검사 | 확인하는 것 |
|---|---|---|
| 1 | test 50명을 **한 명씩 따로** 파싱·변환·예측한 결과 = 한꺼번에 처리한 결과 | test 환자끼리 정보를 주고받지 않는다 |
| 2 | 각 fold의 피처 기준을 정할 때 들어간 ID = 그 fold의 학습 부분 | 단어장이 검증 데이터·test를 보지 않는다 |
| 3 | fold 무결성 (ID 중복 없음, 같은 그룹은 한 fold에만) | 점수가 부풀려지지 않는다 |
| 4 | 제출 형식 (10번 섹션 `check_submission`) | 제출 사고가 나지 않는다 |

In [ ]:
one_by_one = np.vstack([
    np.mean([predict_fold(m, parse_rows(test.iloc[[i]], genes, CFG["HGVS_LEGACY"])) for m in res["models"]], axis=0)
    for i in range(50)])
assert np.allclose(one_by_one, res["te"][:50], atol=1e-6), "검사 1 실패: test 행끼리 영향을 주고 있음"
for k, m in enumerate(res["models"]):
    trn = set(np.where(fold_arr != k)[0])
    assert set(m.fb.fit_ids.tolist()) == trn, f"검사 2 실패: fold {k} 피처 기준에 학습 부분 외의 행이 들어감"
check_folds(folds, y, 26)
print("누수 검사 1~3 통과")

## 8. 결과 분석

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y, res["oof"].argmax(1), target_names=CLASSES, digits=3))
cm = confusion_matrix(y, res["oof"].argmax(1)); np.fill_diagonal(cm, 0)
pairs = sorted(((cm[i, j], CLASSES[i], CLASSES[j]) for i in range(26) for j in range(26) if cm[i, j]), reverse=True)[:10]
print("가장 많이 헷갈린 쌍 (정답 → 예측):")
for c, a, b in pairs:
    print(f"  {a} → {b}: {c}")

### 8-1. 쌍둥이 규칙 (`USE_TWIN`) — 도메인 지식: TCGA 통합 코호트

- TCGA의 **KIPAN**(신장암 통합 = KIRC + KIRP + KICH)과 **GBMLGG**(뇌종양 통합 = GBM + LGG)는 다른 코호트 환자를 **다시 묶은** 코호트다.
  그래서 같은 환자가 `KIRC`로 한 번, `KIPAN`으로 한 번 들어 있다(변이 프로필이 셀 하나까지 똑같음).
- train 안에서 확인: 프로필이 똑같은 그룹 451개 중 (KIPAN, KIRC) 270쌍, (GBMLGG, LGG) 173쌍.
  train에 짝 없이 혼자 있는 KIRC 58명 → test에 같은 프로필 정확히 58명, LGG 52명 → test에 52명. 짝이 test에 있다는 뜻이다.
- 규칙: test 행의 프로필이 train에서 **한 암종으로만** 나온 KIRC/KIPAN/LGG/GBMLGG 프로필과 같으면 **짝 암종**으로 바꾼다
  (KIRC↔KIPAN, LGG↔GBMLGG). 이 네 암종은 서로 거의 구별이 안 돼서 모델 F1이 0.3~0.6이었다.
- 누수 아님: 찾아보는 사전은 **train만으로** 만들고, test 행은 **자기 프로필 하나만** 들고 찾아본다(행 독립, test끼리 비교하지 않음).
- 검증: 우리 fold는 같은 프로필을 한 fold에 묶어서 규칙이 발동할 일이 없다. 그래서 **무작위 5-fold**(대회 train/test 분할 흉내)로
  "짝이 학습 쪽에 있으면 규칙 적용"을 시뮬레이션하고, 점수 비교는 평소처럼 우리 fold별로 한다.

In [ ]:
from sklearn.model_selection import KFold

TWIN_SWAP = {"KIPAN": "KIRC", "KIRC": "KIPAN", "GBMLGG": "LGG", "LGG": "GBMLGG"}


def twin_override(tr_hash, tr_labels, q_hash, classes):
    """질의 행마다 바꿀 암종 번호(없으면 -1). train 프로필 → 암종 집합 사전을 만들고 한 행씩 찾아본다."""
    lab = pd.Series(np.asarray(tr_labels)).groupby(np.asarray(tr_hash)).agg(set).to_dict()
    cls = list(classes)
    out = np.full(len(q_hash), -1)
    for i, h in enumerate(q_hash):
        s = lab.get(h, set())
        if len(s) == 1 and next(iter(s)) in TWIN_SWAP:      # 쌍이 train에 이미 다 있으면(암종 2개) 건드리지 않음
            out[i] = cls.index(TWIN_SWAP[next(iter(s))])
    return out


def apply_override(prob, ov):
    """ov >= 0 인 행은 그 암종 확률을 1로(나머지 0), 나머지 행은 그대로."""
    p = prob.copy(); m = ov >= 0
    p[m] = 0.0; p[m, ov[m]] = 1.0
    return p


def twin_sim(oof, hashes, labels, seed):
    """무작위 5-fold로 '짝이 학습 쪽에 있을 때' 규칙을 OOF에 적용한 결과 (발동 행 수 포함)"""
    out, n = oof.copy(), 0
    for trn, val in KFold(5, shuffle=True, random_state=seed).split(hashes):
        ov = twin_override(hashes[trn], labels[trn], hashes[val], CLASSES)
        out[val] = apply_override(oof[val], ov); n += (ov >= 0).sum()
    return out, n

In [ ]:
_C = np.array(["BRCA", "GBMLGG", "KIPAN", "KIRC", "LGG"])
_ov = twin_override(["a", "a", "b", "c", "d", "e", "e"], ["KIPAN", "KIRC", "KIRC", "LGG", "BRCA", "KIPAN", "KIPAN"],
                    ["a", "b", "c", "d", "z", "e"], _C)
assert _ov.tolist() == [-1, 2, 1, -1, -1, 3]   # a: 쌍이 이미 train에 다 있음 → 그대로 / b: KIRC → KIPAN / c: LGG → GBMLGG
_p = apply_override(np.full((6, 5), 0.2), _ov)
assert (_p[1] == [0, 0, 1, 0, 0]).all() and (_p[0] == 0.2).all() and np.allclose(_p.sum(1), 1)
# 행 독립: 한꺼번에 == 한 행씩
assert all(twin_override(["b"], ["KIRC"], [q], _C)[0] == twin_override(["b"], ["KIRC"], ["b", "z"], _C)[i]
           for i, q in enumerate(["b", "z"]))

In [ ]:
H_tr, H_te = folds.group.values, profile_hashes(test, genes)
assert (H_tr == profile_hashes(train, genes)).all(), "folds.csv의 group이 프로필 지문과 다름"
for seed in (0, 1, 2):
    sim, n = twin_sim(res["oof"], H_tr, train.SUBCLASS.values, seed)
    hit = (sim != res["oof"]).any(1)
    print(f"seed{seed}: 규칙 적용 OOF F1 {macro_f1(y, sim):.4f} (+{macro_f1(y, sim) - res['cv_f1']:.4f}) | "
          f"발동 {n}행 ({n / len(y):.1%}), 발동 행 정답률 {(sim[hit].argmax(1) == y[hit]).mean():.3f} | "
          f"채택 {adopt(sim, res['oof'], y, fold_arr)}")
ov_te = twin_override(H_tr, train.SUBCLASS.values, H_te, CLASSES)
print(f"test에서 발동: {(ov_te >= 0).sum()}행 ({(ov_te >= 0).mean():.1%}) →",
      pd.Series(CLASSES[ov_te[ov_te >= 0]]).value_counts().to_dict())

## 9. 팀 앙상블

팀원은 **우리 `folds.csv`로** 만든 확률 파일을 `team/` 폴더에 넣는다.
- `oof_<이름>.npy` (6201×26, 행 순서 = train.csv), `test_<이름>.npy` (2546×26, 행 순서 = test.csv)
- 열(암종) 순서는 알파벳순, 각 행의 합은 1

파일이 이상하면(크기, NaN, 행 합, 열 순서가 뒤섞임) 섞지 않고 이유를 출력한다.

In [ ]:
def load_member(name, team_dir, y_true):
    """팀원 확률 파일을 검사해서 (oof, test)를 돌려준다. 문제가 있으면 이유를 출력하고 None."""
    try:
        oof = np.load(Path(team_dir) / f"oof_{name}.npy"); te = np.load(Path(team_dir) / f"test_{name}.npy")
    except FileNotFoundError as e:
        print(f"  [{name}] 거부: 파일 없음 ({e.filename})"); return None
    if oof.shape != (len(y_true), len(CLASSES)) or te.shape != (len(test), len(CLASSES)):
        print(f"  [{name}] 거부: 크기 {oof.shape}/{te.shape} (기대 {(len(y_true), 26)}/{(len(test), 26)})"); return None
    if not (np.isfinite(oof).all() and np.isfinite(te).all()):
        print(f"  [{name}] 거부: NaN/inf 포함"); return None
    if not (np.allclose(oof.sum(1), 1, atol=1e-3) and np.allclose(te.sum(1), 1, atol=1e-3)):
        print(f"  [{name}] 거부: 행 합이 1이 아님"); return None
    f = macro_f1(y_true, oof)
    if f < 0.2:   # 정상 모델은 0.35 이상. 이보다 낮으면 열(암종) 순서가 뒤섞였을 가능성이 큼
        print(f"  [{name}] 거부: OOF F1 {f:.3f} < 0.2 (열 순서 확인 필요)"); return None
    return oof, te


def pick_final(cands, y_true, fold_arr):
    """cands: {이름: (oof, te)}. 가장 좋은 단일 모델과 단순 평균을 채택 규칙으로 비교한다."""
    score = {n: macro_f1(y_true, o) for n, (o, _) in cands.items()}
    for n, s in score.items():
        print(f"  {n:12s} OOF F1 = {s:.4f}")
    best = max(score, key=score.get)
    if len(cands) == 1:
        return best, cands[best][1]
    names = list(cands)
    trials = [("평균(" + "+".join(names) + ")", {n: 1.0 for n in names})]
    weakest = min(score, key=score.get)
    trials += [(f"평균({weakest}×{w})", {n: (w if n == weakest else 1.0) for n in names}) for w in (0.5, 0.25)]
    for label, wts in trials:     # 단순 평균이 안 되면, 가장 약한 멤버 비중만 낮춰 본다
        tot = sum(wts.values())
        o = sum(wts[n] * cands[n][0] for n in names) / tot
        ok = adopt(o, cands[best][0], y_true, fold_arr)
        print(f"  {label}: OOF F1 = {macro_f1(y_true, o):.4f}, 최고 단일({best}) 대비 채택 = {ok}")
        if ok:
            return label, sum(wts[n] * cands[n][1] for n in names) / tot
    return best, cands[best][1]


cands = {"v10": (res["oof"], res["te"])}
members = sorted(p.stem[4:] for p in Path(CFG["TEAM_DIR"]).glob("oof_*.npy")) if Path(CFG["TEAM_DIR"]).exists() else []
for n in members:
    got = load_member(n, CFG["TEAM_DIR"], y)
    if got is not None:
        cands[n] = got
if len(cands) == 1:
    print("팀원 파일 없음 → 우리 모델(v10)만 사용")
final_name, final_te = pick_final(cands, y, fold_arr)
print("최종 선택:", final_name)
if final_name != "v10":
    np.save(OUT / "test_ensemble.npy", final_te)

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as d:
    np.save(f"{d}/oof_ok.npy", res["oof"]); np.save(f"{d}/test_ok.npy", res["te"])
    assert load_member("ok", d, y) is not None
    np.save(f"{d}/oof_shape.npy", res["oof"][:100]); np.save(f"{d}/test_shape.npy", res["te"])
    assert load_member("shape", d, y) is None

    def test_member_permuted_columns():            # 열(암종) 순서가 뒤섞인 파일은 거부
        perm = np.roll(np.arange(26), 5)
        np.save(f"{d}/oof_perm.npy", res["oof"][:, perm]); np.save(f"{d}/test_perm.npy", res["te"][:, perm])
        assert load_member("perm", d, y) is None

    test_member_permuted_columns()

## 10. 제출 파일

확률이 가장 높은 암종을 그대로 답으로 쓴다(argmax). 비율 맞추기·승수는 하지 않는다.
예외는 하나: `USE_TWIN`이면 8-1절 쌍둥이 규칙으로 해당 행만 짝 암종으로 바꾼다(train 사전만 사용).

In [ ]:
def make_submission(te_prob, classes, sample):
    sub = sample[["ID"]].copy()
    sub["SUBCLASS"] = classes[te_prob.argmax(1)]
    return sub


def check_submission(sub, sample, classes):
    """누수 검사 4: 제출 형식"""
    assert len(sub) == len(sample) == 2546, len(sub)
    assert (sub.ID.values == sample.ID.values).all(), "ID 순서가 sample_submission과 다름"
    assert set(sub.SUBCLASS) <= set(classes), "train에 없는 암종 이름이 있음"


if CFG["USE_TWIN"]:
    final_te = apply_override(final_te, ov_te)
    print(f"쌍둥이 규칙 적용: test {(ov_te >= 0).sum()}행")
sub = make_submission(final_te, CLASSES, sample)
check_submission(sub, sample, CLASSES)
sub.to_csv(OUT / "submission.csv", index=False, encoding="utf-8-sig")
print(f"저장: {OUT / 'submission.csv'}")
# 참고용 출력만 한다. 이 분포를 보고 아무것도 바꾸지 않는다 (V7 실수 방지).
print("예측 암종 분포:", sub.SUBCLASS.value_counts().to_dict())

### 제출 기록 (LB 점수를 보고 설정을 바꾸지 않는다)

| 날짜 | 실험명 | CV Macro F1 | Public LB | 메모 |
|---|---|---|---|---|
| 11/10 | `gf2_sig` | 0.4956 | 0.4172 | Geneformer V2 + 시그니처 + 쌍둥이 규칙. test 표기 차이로 CV보다 크게 낮음 |